# Week 2, Day 1 — Impurity vs. permutation importance

Day 10 already showed, on one fixed XGBoost model, that gain-based importance and mean-|SHAP| contribution can rank the same features differently. Today pushes on a related but sharper question: two *different* scoring philosophies — impurity-based importance (read straight off the trees, free with every fit) and permutation importance (computed by breaking one feature at a time and watching test accuracy suffer) — compared across three separately-tuned models on one shared feature pipeline. The headline reason they can disagree: correlated features. When two columns carry overlapping information, impurity splits credit between them somewhat arbitrarily (whichever one a greedy split happens to grab), while permutation asks a sharper question — does the model's real test-time accuracy actually depend on this feature, given everything else it still has access to? `pclass` and `fare` are the obvious candidate pair on this dataset (lower class number, higher fare — same socioeconomic signal, two columns), and Step 2 checks that directly before the importance comparison.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

## Setup — one shared pipeline for all three models

Day 7 and Day 8's tree and forest were built on the `who`/`pclass`/`fare`/`family_size` feature set (manual `.map()` encoding). That set can't produce the side-by-side importance table this notebook needs, because it folds `sex` and `age` together into one categorical (`who`) and drops `embarked` entirely. So this notebook retunes the tree and forest from scratch on Day 10's richer pipeline instead — `sex`/`pclass`/`embarked` one-hot encoded via `ColumnTransformer` + `OneHotEncoder(drop="if_binary")`, `age`/`fare`/`sibsp`/`parch` numeric — so all three models see exactly the same columns and a fair comparison is actually possible.

One deliberate difference from Day 10: `age`'s missing values are median-imputed here (`pre_imputed`), not passed through raw (`pre_native`). Plain `DecisionTreeClassifier` and `RandomForestClassifier` don't share XGBoost's native missing-value handling, and today's subject is importance methods, not missingness — keeping all three models on identical, complete-data columns matters more here than it did on Day 10.

In [ ]:
df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "pclass", "embarked"]

pre_imputed = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(drop="if_binary"), categorical_features),
    ]
)

print("train shape:", x_train.shape, " test shape:", x_test.shape)

## Step 1 — retune Tree, Forest and XGBoost on the shared pipeline

Same nested-loop-plus-`cross_val_score` discipline as every prior grid search this course — no `GridSearchCV`. Tree reuses Day 7's `max_depth` grid, forest reuses Day 8's `n_estimators`×`max_depth` grid, XGBoost reuses Day 10's `n_estimators`×`max_depth`×`learning_rate` grid — same ranges, new pipeline underneath.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
tree_results = []
for depth in depths:
    pipe = Pipeline(
        [
            ("pre", pre_imputed),
            ("clf", DecisionTreeClassifier(max_depth=depth, random_state=42)),
        ]
    )
    scores = cross_val_score(pipe, x_train, y_train, cv=5)
    tree_results.append((depth, scores.mean(), scores.std()))
    label = str(depth) if depth is not None else "None"
    print(f"max_depth={label:<4}: CV={scores.mean():.4f} (+/-{scores.std():.4f})")

best_depth = max(tree_results, key=lambda r: r[1])[0]
print(f"\nbest by CV mean: max_depth={best_depth}")

best_tree = Pipeline(
    [
        ("pre", pre_imputed),
        ("clf", DecisionTreeClassifier(max_depth=best_depth, random_state=42)),
    ]
)
best_tree.fit(x_train, y_train)
tree_test_preds = best_tree.predict(x_test)
tree_test_acc = accuracy_score(y_test, tree_test_preds)
print(f"\ntree test accuracy: {tree_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, tree_test_preds))

In [ ]:
n_estimators_grid = [10, 50, 100, 200]
max_depth_grid = [3, 4, 5, 6, 8, None]
forest_results = []
for n_est in n_estimators_grid:
    for depth in max_depth_grid:
        pipe = Pipeline(
            [
                ("pre", pre_imputed),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=n_est, max_depth=depth, random_state=42
                    ),
                ),
            ]
        )
        scores = cross_val_score(pipe, x_train, y_train, cv=5)
        forest_results.append((n_est, depth, scores.mean(), scores.std()))

forest_results.sort(key=lambda r: -r[2])
print("top 5 forest configs by CV mean:")
for n_est, depth, mean, std in forest_results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={str(depth):<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n, best_depth_f, best_cv_f, best_std_f = forest_results[0]
print(
    f"\nbest forest config: n_estimators={best_n}, max_depth={best_depth_f}, CV acc={best_cv_f:.4f} (+/-{best_std_f:.4f})"
)

best_forest = Pipeline(
    [
        ("pre", pre_imputed),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=best_n, max_depth=best_depth_f, random_state=42
            ),
        ),
    ]
)
best_forest.fit(x_train, y_train)
forest_test_preds = best_forest.predict(x_test)
forest_test_acc = accuracy_score(y_test, forest_test_preds)
print(f"\nforest test accuracy: {forest_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, forest_test_preds))

In [ ]:
n_estimators_grid_x = [50, 100, 200]
max_depth_grid_x = [2, 3, 4]
lr_grid = [0.05, 0.1, 0.2]
xgb_results = []
for n_est in n_estimators_grid_x:
    for depth in max_depth_grid_x:
        for lr in lr_grid:
            pipe = Pipeline(
                [
                    ("pre", pre_imputed),
                    (
                        "clf",
                        XGBClassifier(
                            n_estimators=n_est,
                            max_depth=depth,
                            learning_rate=lr,
                            random_state=42,
                            eval_metric="logloss",
                        ),
                    ),
                ]
            )
            scores = cross_val_score(pipe, x_train, y_train, cv=5)
            xgb_results.append((n_est, depth, lr, scores.mean(), scores.std()))

xgb_results.sort(key=lambda r: -r[3])
print("top 5 xgb configs by CV mean:")
for n_est, depth, lr, mean, std in xgb_results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={depth}, lr={lr:<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n_x, best_depth_x, best_lr_x, best_cv_x, best_std_x = xgb_results[0]
print(
    f"\nbest xgb config: n_estimators={best_n_x}, max_depth={best_depth_x}, lr={best_lr_x}, CV acc={best_cv_x:.4f} (+/-{best_std_x:.4f})"
)

best_xgb = Pipeline(
    [
        ("pre", pre_imputed),
        (
            "clf",
            XGBClassifier(
                n_estimators=best_n_x,
                max_depth=best_depth_x,
                learning_rate=best_lr_x,
                random_state=42,
                eval_metric="logloss",
            ),
        ),
    ]
)
best_xgb.fit(x_train, y_train)
xgb_test_preds = best_xgb.predict(x_test)
xgb_test_acc = accuracy_score(y_test, xgb_test_preds)
print(f"\nxgb test accuracy: {xgb_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, xgb_test_preds))

Real numbers against the Week-1 baselines on this same 179-row test set: tree 0.7989 here vs. Day 7's 0.8212 (this pipeline's tree does *worse* — `who` folding sex and age together was a genuinely stronger single feature than sex and age split apart, for a shallow tree with a small depth budget). Forest: 0.8156 here vs. Day 8's 0.8212, a small step down. XGBoost: 0.8212 here vs. Day 10's 0.8156 — the one model that improved, once age's missingness is imputed instead of left native. None of this is the point of today's notebook, but it's worth naming rather than skating past: changing the feature representation changes accuracy too, not just importance rankings — the two aren't independent.

In [ ]:
tree_names = list(best_tree.named_steps["pre"].get_feature_names_out())
forest_names = list(best_forest.named_steps["pre"].get_feature_names_out())
xgb_names = list(best_xgb.named_steps["pre"].get_feature_names_out())
assert tree_names == forest_names == xgb_names
print("shared feature columns:", tree_names)

## Step 2 — checking the multicollinearity claim, not assuming it

`pclass` and `fare` are the obvious redundant pair here — cabin class and ticket price are two measurements of roughly the same thing (socioeconomic status). Before leaning on that in Step 5, check it directly rather than asserting it: a simple correlation between `pclass` (1/2/3) and `fare`, plus mean fare by class.

In [ ]:
corr = df[["pclass", "fare"]].corr().iloc[0, 1]
print(f"corr(pclass, fare) = {corr:.4f}")
print()
print(df.groupby("pclass")["fare"].mean())

`corr(pclass, fare) = -0.5495` — moderately strong, and in the expected direction (lower class *number* = higher fare, since 1st class is the priciest). Mean fare by class makes the redundancy concrete: 1st class averages 84.15, 2nd 20.66, 3rd 13.68 — a passenger's fare is already telling the model most of what `pclass` would. That's the setup for Step 5: if a model can reconstruct most of `pclass`'s signal from `fare` alone (or vice versa), permuting one of them shouldn't hurt much, even if it earned real credit at split time.

## Step 3 — impurity importance, side by side

`.named_steps["clf"].feature_importances_` from each of the three tuned pipelines, collapsed from one-hot columns back to their parent feature (`cat__pclass_1` + `cat__pclass_2` + `cat__pclass_3` → `pclass`, same for `sex` and `embarked`) so the three models' importances line up on the same seven rows.

In [ ]:
def parent_feature(name):
    name = name.split("__", 1)[1]
    for base in ["pclass", "sex", "embarked"]:
        if name.startswith(base + "_"):
            return base
    return name


models = {"tree": best_tree, "forest": best_forest, "xgboost": best_xgb}
imp_table = {}
for label, model in models.items():
    names = model.named_steps["pre"].get_feature_names_out()
    imps = model.named_steps["clf"].feature_importances_
    agg = {}
    for n, v in zip(names, imps):
        p = parent_feature(n)
        agg[p] = agg.get(p, 0) + v
    imp_table[label] = agg

parents = sorted(set(p for t in imp_table.values() for p in t))
print(f"{'feature':<12}" + "".join(f"{m:>10}" for m in models))
for p in parents:
    row = "".join(f"{imp_table[m].get(p, 0):>10.4f}" for m in models)
    print(f"{p:<12}{row}")

`sex` and `pclass` lead all three models — the same "women and children first, and class mattered" signal Day 1-2's EDA, Day 6's logistic regression, and Day 7's tree already converged on independently. The one number worth flagging before Step 4: the random forest gives `fare` **0.2091** — second place, ahead of `pclass`'s 0.1259, and not far behind `sex`. Tree and XGBoost don't rate `fare` nearly that highly (0.0612 and 0.0400). That's the exact kind of single-model outlier Step 2's correlation finding predicts trouble for — watch what happens to the forest's `fare` number specifically once permutation gets a turn.

## Step 4 — permutation importance, on the test set only

```python
from sklearn.inspection import permutation_importance
perm = permutation_importance(model, x_test, y_test, n_repeats=30, random_state=42, scoring="accuracy")
```

Two choices worth explaining, not just making silently:

**Test set, not train.** The question permutation importance answers is "how much does the model's *real generalization performance* depend on this feature" — that's a test-set question, the mirror image of every CV-on-train-only discipline this course has followed since Day 7 (there, train-only CV picks hyperparameters honestly; here, test-only permutation scores honestly).

**Permuting the raw column, before encoding — not the one-hot dummy after encoding.** `permutation_importance` is called on the *whole pipeline* (`pre_imputed` + classifier) with the raw `x_test`, so it shuffles `pclass`'s three raw values (1/2/3) together as one column, then re-encodes. Shuffling a single dummy column after encoding (e.g. `cat__pclass_3` alone, independent of `cat__pclass_1`/`cat__pclass_2`) would create rows with two class dummies lit at once, or none at all — combinations that never occur in real data and that the model was never trained to handle. That's not a hypothetical concern: doing it that way on this model shows `cat__pclass_3` alone scoring +0.0842 for XGBoost, nearly as high as `pclass`'s honest combined permutation score of 0.1048 — a number that looks meaningful but is partly an artifact of feeding the model nonsense rows, not a clean read of `pclass`'s real importance.

In [ ]:
raw_columns = list(x_test.columns)
print("raw columns:", raw_columns)
print()

for label, model in models.items():
    perm = permutation_importance(
        model, x_test, y_test, n_repeats=30, random_state=42, scoring="accuracy"
    )
    order = np.argsort(-perm.importances_mean)
    print(f"=== {label} ===")
    for i in order:
        print(
            f"  {raw_columns[i]:<10}: {perm.importances_mean[i]:+.4f} (+/-{perm.importances_std[i]:.4f})"
        )
    print()

## Step 5 — the reveal, in this notebook's own numbers

The clean version of the effect shows up in exactly the model Step 3 flagged: the **random forest**. `fare` goes from **0.2091** (2nd place, ahead of `pclass`) in impurity importance to **0.0136** (5th place, barely above the two weakest features) in permutation importance — a 15x drop, and a real reordering, not noise (`fare`'s permutation std is 0.0129, comfortably smaller than the 0.042 point-gap it needs to close to retake 2nd place from `age`). `pclass`, meanwhile, holds essentially the same standing in both measures for the forest (4th by impurity, 2nd by permutation — actually *moves up*).

The mechanism is exactly Step 2's finding: `fare` and `pclass` carry overlapping socioeconomic signal (`corr=-0.5495`). The forest's greedy splits found `fare` a genuinely useful column to split on early and often — hence the high impurity credit — but once `fare` is shuffled away, the forest still has `pclass` sitting right there carrying most of the same information, so test accuracy barely moves. Impurity importance measures "how much did this feature get used," which doesn't distinguish "used because irreplaceable" from "used because it happened to be available and correlated with something irreplaceable." Permutation importance measures the second thing directly, by actually taking the feature away and checking.

Tree and XGBoost show a smaller version of the same story, worth reporting honestly rather than inflating: the tree's `fare` importance was already modest by impurity (0.0612, 4th place, behind `age`) and stays at 4th under permutation (0.0160) — no reordering at all for the tree, because a depth-3 tree has few splits to spend and never leaned on `fare` heavily enough for permutation to have anything dramatic to reveal. XGBoost's `fare` actually holds its relative rank (5th impurity, 4th permutation) — its splits already favored `pclass` (0.3228 impurity, the highest of any model) over `fare`, so there was less redundant credit sitting on `fare` to begin with. The lesson isn't "fare is unimportant" or "pclass always wins" — it's that *which* correlated feature ends up over-credited by impurity depends on which one the model's specific greedy splits happened to grab first, and only permutation importance, run on held-out data, catches it after the fact.

## Step 6 — a fourth angle: L2 logistic regression coefficients

One more independent read on the same question, from a model family with no impurity concept at all. `LogisticRegression`'s default penalty is L2 (Day 6 already built this from scratch and derived why); fit here on the identical `pre_imputed` pipeline so the coefficients are on the same one-hot columns as everything above.

In [ ]:
logreg = Pipeline([("pre", pre_imputed), ("clf", LogisticRegression(max_iter=1000))])
logreg.fit(x_train, y_train)
logreg_test_preds = logreg.predict(x_test)
logreg_test_acc = accuracy_score(y_test, logreg_test_preds)

names = logreg.named_steps["pre"].get_feature_names_out()
coefs = logreg.named_steps["clf"].coef_[0]

print(f"logistic regression test accuracy: {logreg_test_acc:.4f}\n")
for n, c in sorted(zip(names, coefs), key=lambda t: -abs(t[1])):
    print(f"  {n:<20}: {c:+.4f}")

### A scale caveat, before reading these as "importances"

`fare` ranges from 0 to 512 in the training data (std ≈ 52); `sex_male` only ever takes the value 0 or 1. A logistic regression coefficient is "change in log-odds per one-unit increase" — for `fare` that's the effect of one extra *dollar*, for `sex_male` that's the effect of the entire category swing. Comparing those two raw coefficients directly, like the printout above does, isn't a fair comparison — `fare`'s coefficient looks tiny partly because one dollar is a tiny fraction of its actual range, not only because `fare` carries little information.

Fix: rescale each numeric coefficient by that feature's own standard deviation, so every number reads as "effect of one typical (1-std) swing in this feature" — directly comparable to a dummy's coefficient, which is already the effect of its one possible swing (0 to 1).

In [ ]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
numeric_stds = x_train[numeric_cols].fillna(x_train[numeric_cols].median()).std()

scaled = []
for n, c in zip(names, coefs):
    base = n.split("__", 1)[1]
    effect = c * numeric_stds[base] if base in numeric_stds.index else c
    scaled.append((n, c, effect, base in numeric_stds.index))

print("per-typical-swing effect (dummy coefficients are already a full 0->1 swing):\n")
for n, c, effect, is_numeric in sorted(scaled, key=lambda t: -abs(t[2])):
    tag = f"x std={numeric_stds[n.split('__', 1)[1]]:.2f}" if is_numeric else "(dummy)"
    print(f"  {n:<20}: raw={c:+.4f}  {tag:<14} -> per-typical-swing={effect:+.4f}")

Coefficient magnitude broadly echoes permutation importance's story, once compared fairly. `sex_male` (-2.5763) and the `pclass` dummies (`pclass_3` -0.9666, `pclass_1` +0.7286) carry by far the largest per-swing effects — consistent with `sex` and `pclass` dominating both impurity and permutation importance across all three tree ensembles. Rescaled to a per-typical-swing basis (previous cell), `fare`'s effect is **+0.19** — 9th of 12, clearly behind `sex`, all three `pclass` dummies, `age`, and `sibsp`. Not literally the smallest coefficient in the model, though (that's `embarked_Q` at -0.0013, then `parch` at -0.10) — the raw, unscaled coefficient (+0.0037) looked far smaller than that only because `fare`'s dollar-valued scale makes a "one-unit" swing tiny relative to its actual range. Corrected for scale, `fare`'s effect is modest but not negligible — still a fourth, structurally independent read that lines up with Step 5 (once `pclass` is already in the model, `fare` has comparatively little *additional* linear signal to contribute), just less dramatically "near zero" than the raw coefficient alone suggested.

The through-line for today, across four different scoring methods on three different model families: correlated features don't have one "true" importance that every method agrees on. Impurity importance and raw coefficient magnitude can both mislead — impurity by crediting whichever correlated feature the fitting process happened to lean on, raw coefficients by conflating a feature's information content with its measurement scale. Permutation importance is the one method here that sidesteps both: shuffle the raw feature, keep everything else fixed, and check whether the model's real accuracy actually notices. On this dataset, for the random forest specifically, `fare` is the feature that answer turns out to be "no" for.